# Three regimes of the period-1 branch vs $\kappa_b$: limit cycles, bifurcation diagrams and chaos (22/10/2026)

Follow-up of `Hill_method_satellite_fold.ipynb` (section "Beyond the cusp at large $g_2$"). At $\varepsilon_d = 10.12\,g_2$ the period-1 branch (cat, saddle, satellite) has three regimes in $\kappa_b$:

| regime | $\kappa_b/\kappa_b^{(0)}$ | example here |
|---|---|---|
| **A**: below the cusp, with period doubling of the satellite | $< 5.36$ | $\kappa_b^{(0)}$ |
| **B**: below the cusp, no period doubling | $5.36$–$9.70$ | $8\,\kappa_b^{(0)}$ |
| **C**: beyond the cusp (single period-1 branch) | $> 9.70$ | $11\,\kappa_b^{(0)}$ |

Sections:
1. Regime diagram in the $(\kappa_b, g_2)$ plane (the figure of `Hill_method_satellite_fold`, with the Neimark–Sacker line of the period-2 orbit added).
2. Limit cycles of the three regimes (harmonic balance and Floquet multipliers), and the period-2 orbit of regime A.
3. Bifurcation diagrams (attractor followed in $g_2$ by time integration, as in `torus_breakdown.ipynb`), and, along one sweep per regime, the largest Lyapunov exponent.
4. Map of the type of attractor (period on the stroboscopic section) in the $(\kappa_b, g_2)$ plane.

**Conventions.** $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\varepsilon_d = 10.12\,g_2$, $\kappa_b^{(0)} = \pi/10.4$ in the code convention ($-2\kappa_b\dot\phi_b$; $\kappa_b^{(0)}/2\pi = 96$ MHz in the article's GKSL convention).
$\tau = \omega_a t$, memory period $T_a = 2\pi/\omega_a$. The physical range ends at $g_2^{\max} = E_J\varphi_a^2\varphi_b = 137.4$ MHz ($\varepsilon_p = \pi/2$).
Above it the computation is formal: $s = \sin\varepsilon_p > 1$, i.e. the same model with a larger $E_J\varphi_a^2\varphi_b$ (shaded grey in the figures).

The time integrations (sections 3 and 4) are done in Julia by `beyond_cusp_dynamics.jl` (same equations as `catchaos.ats.full_rhs`), called from this notebook.

In [ ]:
import json, os, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.optimize import root
from scipy.special import jv
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/pylib")
from catchaos.params import *                      # w_a, w_b, phi_a, phi_b, E_J, kappa_b (kappa_b0, code convention), G0, MHz, g2_max
from catchaos.plotting import *                    # figsize, figsize_car, Perso_palette_float

HERE = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/studies/torus_crisis/"
OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/torus_crisis/"

r0 = 10.12
K_REG = {"A": 1.0, "B": 8.0, "C": 11.0}                          # kappa_b / kappa_b0 of the three examples
REG_NAME = {"A": "below the cusp, period doubling", "B": "below the cusp, no period doubling", "C": "beyond the cusp"}

## Harmonic balance, Floquet multipliers, analytic seed
From `catchaos`: `ats.HB` (period 1, exact one-period AFT) and `ats.HB2` (period 2: $\tau_d = \tau/2$, $M\to M/4$, $C\to C/2$, drive on harmonic 4), built with `kfac` ($\kappa_b/\kappa_b^{(0)}$) and `r = r0`;
`hill.monodromy` (multipliers from the monodromy matrix, over $T_a$ or $2T_a$ from `hb.period`), `hill.instability`, `strobe`, `memory_phase`, `odd_content`, `ats.rotating_frame`;
`seeds.satellite_hb`, `seeds.seed_z` (analytic satellite); `hill.extended`, `hill.tangent` (arclength in section 2).

In [ ]:
from catchaos.ats import n, HB, HB2, rotating_frame
from catchaos.hill import strobe, memory_phase, odd_content, monodromy, instability, extended, tangent
from catchaos.seeds import satellite_hb, seed_z

## Seeds
- Period 1: the satellite at 60 MHz from the analytic seed, continued to 120 MHz, then in $\kappa_b$ at 120 MHz (`orbit_at_kappa`).
- Period 2: the doubled satellite of `Hill_method_doubling_corrected.ipynb` at $\varepsilon_p = 1.2$ (128 MHz), converged with the former $J_1$ definition of $g_2$ and continued in $\varepsilon_d$ to the $\sin$ definition, $N_H = 24$.

In [ ]:
N_H, N_H2 = 16, 24
hb = HB(N_H, r=r0)
s60 = 60/g2_max
hb.set_s(s60)
z_sat60 = hb.solve(seed_z(satellite_hb(s60, r0), N_H))
s_hi = 120/g2_max
z_hi = z_sat60.copy()
for s in np.linspace(s60, s_hi, 25)[1:]:
    hb.set_s(s); z_hi = hb.solve(z_hi)

def orbit_at_kappa(kfac, z=None, k_from=1.0):
    # period-1 orbit at 120 MHz for kappa_b x kfac, continued in kappa_b
    z = z_hi.copy() if z is None else z.copy()
    for kk in np.linspace(k_from, kfac, 1 + int(np.ceil(abs(kfac - k_from)/0.1)))[1:]:
        h_ = HB(N_H, kfac=kk, r=r0); h_.set_s(s_hi); z = h_.solve(z)
    return z

In [ ]:
z2_seed = np.array([-8.67480681e-02, -1.49178000e-01, -1.55487349e-04, -3.92476650e-06,
         4.80123540e-02,  6.60523141e-02, -3.00220194e-02,  3.07073460e-16,
         3.31363194e-01, -1.72222278e-16, -3.88737312e-03,  1.57262610e-02,
         3.35330370e-02, -1.65080648e-01,  6.93437379e-04, -5.59462961e-02,
        -4.67601081e-03,  3.00083819e-01,  4.52017869e-05,  9.32575051e-04,
        -4.88538089e-04, -7.81645147e-03, -1.26778042e-03, -1.46772441e-02,
         4.61868963e-03,  5.06227939e-02,  2.68514538e-04,  2.61445320e-03,
        -1.01232565e-03, -9.46918403e-03, -5.89963411e-04, -5.17649744e-03,
         1.52354566e-03,  1.30588333e-02])     # period-2 orbit at eps_p = 1.2, J1 definition
s_top = np.sin(1.2)
hb2 = HB2(N_H2, r=r0)
z2_top = hb2.pad(z2_seed)
for s_eff in np.linspace(jv(1, 1.2), s_top, 25):          # eps_d = r0 s_eff G0: J1 -> sin
    hb2.set_s(s_top, r0*s_eff/s_top); z2_top = hb2.solve(z2_top)
print(f"period 1 at 120 MHz: max|mu| = {np.abs(monodromy(hb, z_hi)).max():.3f};  "
      f"period 2 at {MHz(s_top*G0):.1f} MHz: max|mu| = {np.abs(monodromy(hb2, z2_top)).max():.2f}, odd harmonics {odd_content(z2_top, N_H2):.3f}")

# 1. Regime diagram
Lines from `beyond_cusp_bifurcations_22_10_2026.json` (`Hill_method_satellite_fold.ipynb`): satellite and cat folds, cusp, period-doubling bubble of the satellite, Neimark–Sacker of the period-1 orbit (formal, above $g_2^{\max}$).
Added here: the **Neimark–Sacker of the period-2 orbit**, inside the bubble. For each $\kappa_b$ the period-2 orbit is continued in $\kappa_b$ at 128 MHz, then down in $g_2$ (steps of 0.2 MHz) until it becomes stable;
the crossing of $\max|\mu| = 1$ is interpolated linearly. Below that point it is stable down to the period doubling where it is born.
The three examples A, B, C are the vertical dotted lines.

In [ ]:
BIF = json.load(open(OUT_DIR + "hill_orbits/beyond_cusp_bifurcations_22_10_2026.json"))
fl = BIF["fold_lines"]
k_fold = np.array(fl["kappa_over_kappa0"])
g_sat_fold = np.array([np.nan if v is None else v for v in fl["satellite_fold_MHz"]], float)
g_cat_fold = np.array([np.nan if v is None else v for v in fl["cat_fold_MHz"]], float)
k_bub = np.array(BIF["kappa_over_kappa0"])
bubble = BIF["pd_bubble_MHz"]
k_pd, g_pd, k_c, g_c = BIF["kappa_pd_close"], BIF["g2_pd_close_MHz"], BIF["kappa_cusp"], BIF["g2_cusp_MHz"]
k_ns1 = sorted(float(k) for k in BIF["formal"])
g_ns1 = [BIF["formal"][str(int(k)) if str(int(k)) in BIF["formal"] else str(k)]["g2"] for k in k_ns1]

In [ ]:
def p2_branch_at_kappa(kfac, g_lo=60.0, g_hi=180.0, dg=0.2, with_mu=False):
    # period-2 branch (dict g2 -> z), from 128 MHz down to the period doubling (odd harmonics -> 0) and up to g_hi
    z = z2_top.copy()
    for kk in np.linspace(1.0, kfac, 1 + int(np.ceil(abs(kfac - 1)/0.05)))[1:]:
        h_ = HB2(N_H2, kfac=kk, r=r0); h_.set_s(s_top); z = h_.solve(z)
    hbk = HB2(N_H2, kfac=kfac, r=r0); out, end = {}, {}
    for direction, g_end in ((-1, g_lo), (+1, g_hi)):
        zz, g = z.copy(), MHz(s_top*G0)
        while (g_end - g)*direction > 0:
            g_new = g + direction*dg
            hbk.set_s(g_new/g2_max)
            try:
                zn = hbk.solve_robust(zz)
            except RuntimeError:
                end[direction] = f"solver failed at {g_new:.1f} MHz"; break
            if odd_content(zn, N_H2) < 1e-4:
                end[direction] = f"merges with the period-1 orbit at {g_new:.1f} MHz"; break
            zz, g = zn, g_new
            out[round(g, 2)] = (zz.copy(), monodromy(hbk, zz) if with_mu else None)
    return dict(sorted(out.items())), end

def ns_p2(kfac):
    br, end = p2_branch_at_kappa(kfac, g_hi=MHz(s_top*G0), with_mu=True)
    g = np.array(list(br)); m = np.array([np.abs(br[x][1]).max() for x in g])
    i = np.where((m[:-1] < 1) & (m[1:] > 1))[0]
    return (g[i[0]] + (1 - m[i[0]])*(g[i[0] + 1] - g[i[0]])/(m[i[0] + 1] - m[i[0]]) if len(i) else np.nan), end.get(-1, "reaches 60 MHz")

In [ ]:
k_ns2 = [1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5, 5.25]
g_ns2 = []
for k in k_ns2:
    g, end = ns_p2(k)
    g_ns2.append(g)
    print(f"kappa_b x {k:5.2f}: period-2 orbit {end}; Neimark-Sacker at g2/2pi = {g:.2f} MHz")
g_ns2 = np.array(g_ns2)

In [ ]:
C_CAT, C_SAT, C_PD, C_NS = Perso_palette_float[2], Perso_palette_float[5], Perso_palette_float[3], Perso_palette_float[4]

def regime_axes(ax, legend=True, lw=1.0, fills=True):
    ok = np.isfinite(g_sat_fold)
    if fills:
        ax.fill_between(k_fold[ok], g_sat_fold[ok], g_cat_fold[ok], color="0.88", lw=0, label="cat and satellite coexist")
    ax.plot(k_fold[ok], g_sat_fold[ok], color=C_SAT, lw=lw, label="satellite fold")
    ax.plot(k_fold[ok], g_cat_fold[ok], color=C_CAT, lw=lw, label="cat fold")
    ax.plot(k_c, g_c, "k*", ms=6, label="cusp")
    kb = [k for k, b in zip(k_bub, bubble) if len(b) >= 2]
    lo, hi = [b[0] for b in bubble if len(b) >= 2] + [g_pd], [b[-1] for b in bubble if len(b) >= 2] + [g_pd]
    if fills:
        ax.fill_between(kb + [k_pd], lo, hi, color=C_PD, alpha=0.45, lw=0, label="satellite period-doubled")
    else:
        ax.plot(kb + [k_pd] + kb[::-1], lo + hi[-2::-1], "--", color=C_PD, lw=lw, label="period-doubling bubble of the satellite")
    ax.plot(k_ns2, g_ns2, "-", color=C_NS, lw=lw, label="Neimark–Sacker of the period-2 orbit")
    ax.plot(k_ns1, g_ns1, "o--", ms=2, color=C_NS, lw=0.8*lw, label=r"Neimark–Sacker of the period-1 orbit (formal)")
    if fills:
        ax.axhspan(g2_max, 400, color="0.6", alpha=0.15, lw=0)
    ax.axhline(g2_max, color="gray", lw=0.8, label=r"$g_2^{\max} = E_J\varphi_a^2\varphi_b$")
    ax.set_xlabel(r"$\kappa_b/\kappa_b^{(0)}$  ($\kappa_b^{(0)}/2\pi = 96$ MHz)"); ax.set_ylabel(r"$g_2/2\pi$ (MHz)")
    if legend:
        ax.legend(fontsize=5, loc="center left", bbox_to_anchor=(1, 0.5))

fig, ax = plt.subplots(figsize=(1.5*figsize[0], figsize[1]))
regime_axes(ax)
for name, k in K_REG.items():
    ax.axvline(k, ls=":", color="k", lw=0.7)
    ax.text(k + 0.15, 25, name, fontsize=7)
ax.set_xlim(0, 20.5); ax.set_ylim(20, 220)
plt.tight_layout()
plt.savefig(FIG_DIR + "beyond_cusp_regimes_diagram_22_10_2026.pdf", bbox_inches="tight")
plt.show()

# 2. Limit cycles of the three regimes
Same figure as "Limit cycles below and beyond the cusp" in `Hill_method_satellite_fold.ipynb`, extended to the whole $g_2$ range (8–180 MHz, formal above $g_2^{\max}$).
The period-1 branch is followed by arclength from 120 MHz down to 8 MHz (through the S below the cusp), and by natural continuation from 120 up to 180 MHz.
Left: $\langle|a_R|^2\rangle$ along the branch, coloured by the type of the leading multiplier (stable; saddle, $\mu > 1$; period doubling, $\mu < -1$; Neimark–Sacker, complex pair).
Middle and right: orbits along the branch in the rotating frame ($a_R = i\,a\,e^{i\tau}$, $b_R = b\,e^{2i\tau}$), coloured by $g_2$, dashed when unstable; the partner $a\to-a$ is lighter.
The last row is the **period-2 orbit of regime A** (period $2T_a$), with the period-1 branch in grey for reference.

In [ ]:
def arclength_down(kfac, z_start, s_start, g_min=8.0, ds_max=0.01):
    # pseudo-arclength in s from (z_start, s_start) towards lower g2, down to g_min; returned from low g2 up
    hbk = HB(N_H, kfac=kfac, r=r0)
    F, dF = extended(hbk, hbk.set_s)
    y = np.append(z_start, s_start); t = tangent(dF, y, direction=-1); ds = 0.003; hist = [y.copy()]
    for _ in range(10000):
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        if not sol.success:
            ds /= 2
            if ds < 1e-7:
                break
            continue
        y = sol.x; t = tangent(dF, y, t); hist.append(y.copy()); ds = min(1.2*ds, ds_max)
        if MHz(y[-1]*G0) < g_min:
            break
    return np.array(hist[::-1])

In [ ]:
def period1_branch(kfac, g_top=180.0):
    z120 = orbit_at_kappa(kfac)
    down = arclength_down(kfac, z120, s_hi)                    # ordered from low g2 to 120 MHz
    hbk = HB(N_H, kfac=kfac, r=r0); up, z = [], z120.copy()
    for g in np.arange(120.5, g_top + 1e-9, 0.5):
        hbk.set_s(g/g2_max); z = hbk.solve_robust(z); up.append(np.append(z, g/g2_max))
    br = np.vstack([down, np.array(up)])
    idx = np.arange(0, len(br), 2)
    mu = []
    for i in idx:
        hbk.set_s(br[i, -1]); mu.append(monodromy(hbk, br[i, :-1]))
    t = np.linspace(0, 2*np.pi, 300, endpoint=False)
    a2 = np.array([np.mean(np.abs(rotating_frame(br[i, :-1], N_H, t)[0])**2) for i in idx])
    return dict(branch=br, idx=idx, g2=MHz(br[idx, -1]*G0), mu=np.array(mu), kind=np.array([instability(m) for m in mu]), a2=a2)

In [ ]:
P1 = {}
for name, k in K_REG.items():
    P1[name] = period1_branch(k)
    d = P1[name]
    changes = [(d["g2"][i], d["kind"][i - 1], d["kind"][i]) for i in range(1, len(d["kind"])) if d["kind"][i] != d["kind"][i - 1]]
    print(f"regime {name} (kappa_b x {k:g}): " + "; ".join(f"{a} -> {b} at {g:.1f} MHz" for g, a, b in changes))

# period-2 orbit of regime A
P2_br, P2_end = p2_branch_at_kappa(K_REG["A"], with_mu=True)
P2_g = np.array(list(P2_br))
P2_kind = np.array([instability(P2_br[g][1]) for g in P2_g])
t2 = np.linspace(0, 4*np.pi, 600, endpoint=False)
P2_a2 = np.array([np.mean(np.abs(rotating_frame(P2_br[g][0], N_H2, t2, period=2)[0])**2) for g in P2_g])
changes = [(P2_g[i], P2_kind[i - 1], P2_kind[i]) for i in range(1, len(P2_kind)) if P2_kind[i] != P2_kind[i - 1]]
print(f"period-2 orbit, regime A: {P2_g[0]:.1f}-{P2_g[-1]:.1f} MHz ({P2_end.get(-1, 'reaches 60 MHz')}); " + "; ".join(f"{a} -> {b} at {g:.1f} MHz" for g, a, b in changes))

In [ ]:
KIND_STYLE = {"stable": dict(color="k", ls="-"), "saddle": dict(color="0.55", ls="--"),
              "PD": dict(color=C_PD, ls="--"), "NS": dict(color=C_NS, ls="--")}
KIND_LABEL = {"stable": "stable", "saddle": r"saddle ($\mu > 1$)", "PD": r"period doubling ($\mu < -1$)", "NS": "Neimark–Sacker (complex pair)"}
cmap_g2, norm_g2 = plt.get_cmap("viridis"), mpl.colors.Normalize(0, 180)

def plot_kinds(ax, g, y, kind, **kw):
    for k_, st in KIND_STYLE.items():
        m = kind == k_
        if m.any():
            ax.plot(g, np.where(m, y, np.nan), lw=1, **st, **kw)

def plot_orbits(ax_a, ax_b, curves):
    # curves: list of (g2, a_R, b_R, stable)
    lim_a = lim_b = 0
    for g, a_R, b_R, st in curves:
        col = cmap_g2(norm_g2(g))
        for sign, alpha in ((1, 1.0), (-1, 0.3)):
            ax_a.plot(sign*a_R.real, sign*a_R.imag, "-" if st else "--", color=col, lw=0.7, alpha=alpha)
        ax_b.plot(b_R.real, b_R.imag, "-" if st else "--", color=col, lw=0.7)
        lim_a, lim_b = max(lim_a, np.abs(a_R).max()), max(lim_b, np.abs(b_R).max())
    for ax, lab, lim in ((ax_a, "a", lim_a), (ax_b, "b", lim_b)):
        lim = 1.05*lim
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
        ax.set_xlabel(rf"Re ${lab}_R$"); ax.set_ylabel(rf"Im ${lab}_R$")

In [ ]:
t1 = np.linspace(0, 2*np.pi, 300)
fig, axes = plt.subplots(4, 3, figsize=(3*figsize_car[0], 4*figsize_car[1]))
ROW = {"A": 0, "B": 2, "C": 3}                      # row 1: period-2 orbit of regime A
for name, k in K_REG.items():
    row = ROW[name]
    d = P1[name]
    ax = axes[row, 0]
    plot_kinds(ax, d["g2"], d["a2"], d["kind"])
    ax.axvspan(g2_max, 185, color="0.6", alpha=0.15, lw=0)
    ax.set_xlim(0, 182); ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\langle|a_R|^2\rangle$")
    ax.set_title(rf"{name}: $\kappa_b = {k:g}\,\kappa_b^{{(0)}}$, {REG_NAME[name]}", fontsize=6)
    br = d["branch"]
    curves = []
    for i in np.linspace(0, len(br) - 1, 16).astype(int):
        hbk = HB(N_H, kfac=k, r=r0); hbk.set_s(br[i, -1])
        a_R, b_R = rotating_frame(br[i, :-1], N_H, t1)
        curves.append((MHz(br[i, -1]*G0), a_R, b_R, np.abs(monodromy(hbk, br[i, :-1])).max() < 1))
    plot_orbits(axes[row, 1], axes[row, 2], curves)
# period-2 orbit of regime A
ax = axes[1, 0]
dA = P1["A"]
ax.plot(dA["g2"], dA["a2"], color="0.75", lw=1)
plot_kinds(ax, P2_g, P2_a2, P2_kind)
ax.axvspan(g2_max, 185, color="0.6", alpha=0.15, lw=0)
ax.set_xlim(0, 182); ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\langle|a_R|^2\rangle$ (over $2T_a$)")
ax.set_title(r"A: period-2 orbit (grey: period-1 branch)", fontsize=6)
curves = []
for g in P2_g[np.linspace(0, len(P2_g) - 1, 12).astype(int)]:
    a_R, b_R = rotating_frame(P2_br[g][0], N_H2, np.linspace(0, 4*np.pi, 600), period=2)
    curves.append((g, a_R, b_R, np.abs(P2_br[g][1]).max() < 1))
plot_orbits(axes[1, 1], axes[1, 2], curves)
handles = [mpl.lines.Line2D([], [], lw=1, **st) for st in KIND_STYLE.values()]
axes[0, 0].legend(handles, [KIND_LABEL[k] for k in KIND_STYLE], fontsize=4.5)
fig.subplots_adjust(hspace=0.5, wspace=0.45)
fig.colorbar(mpl.cm.ScalarMappable(norm=norm_g2, cmap=cmap_g2), ax=axes[:, 1:], label=r"$g_2/2\pi$ (MHz)", shrink=0.5)
plt.savefig(FIG_DIR + "beyond_cusp_regimes_limit_cycles_22_10_2026.pdf", bbox_inches="tight")
plt.show()

### Floquet multipliers of the period-2 orbit (regime A)
$\max|\mu|$ and the angle of the leading pair (per $2T_a$). The Neimark–Sacker point is where a complex pair leaves the unit circle; its angle is the rotation number expected on the torus
(`torus_breakdown.ipynb`: 0.106 at the onset).

In [ ]:
mu2 = np.array([P2_br[g][1] for g in P2_g])
lead = mu2[np.arange(len(P2_g)), np.argmax(np.abs(mu2), axis=1)]
i_ns = np.where((np.abs(lead[:-1]) < 1) & (np.abs(lead[1:]) > 1))[0][0]
g_ns_A = np.interp(1, np.abs(lead[i_ns:i_ns + 2]), P2_g[i_ns:i_ns + 2])
print(f"period-2 orbit, regime A: born at {P2_g[0]:.1f} MHz, Neimark-Sacker at {g_ns_A:.2f} MHz with arg mu/2pi = {abs(np.angle(lead[i_ns + 1]))/(2*np.pi):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(3*figsize_car[0], figsize_car[1]))
ax = axes[0]
ax.plot(P2_g, np.abs(lead), color="k", lw=1)
ax.axhline(1, ls="--", color="r", lw=0.8); ax.axvline(g_ns_A, ls=":", color="k", lw=0.8)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\max|\mu|$ (period 2)")
ax = axes[1]
ax.plot(P2_g, np.abs(np.angle(lead))/(2*np.pi), color="k", lw=1)
ax.axvline(g_ns_A, ls=":", color="k", lw=0.8)
ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$|\arg\mu|/2\pi$, leading multiplier")
ax = axes[2]
th = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(th), np.sin(th), "k-", lw=0.5)
w = P2_g < 90
sc = ax.scatter(mu2[w].real.ravel(), mu2[w].imag.ravel(), c=np.repeat(P2_g[w], 4), s=2, cmap="viridis")
plt.colorbar(sc, ax=ax, label=r"$g_2/2\pi$ (MHz)")
ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.6, 1.6); ax.set_aspect("equal")
ax.set_xlabel(r"Re $\mu$"); ax.set_ylabel(r"Im $\mu$"); ax.set_title(r"$g_2 < 90$ MHz", fontsize=6)
plt.tight_layout()
plt.show()

# 3. Bifurcation diagrams
`beyond_cusp_dynamics.jl` follows the attractor in $g_2$ by parameter continuation, in steps of 0.2 MHz: each $g_2$ starts from the end state of the previous one, followed by $2000\,T_a$ of transient,
then the state is recorded over $400\,T_a$ **at every drive period** $T_a/2$ (the last 160 samples, i.e. $80\,T_a$, are kept for the diagram), and the period on the section at $T_a$ is detected (up to 64; 0 if none: torus or chaos).

**Why the drive period.** The two partners $\pm a$ of a period-1 orbit are the same orbit shifted by one drive period ($\tau \to \tau + \pi$; in the rotating frame $a_R \to -a_R$).
On the section at $T_a$ they give different points, $a(0)$ and $a(\pi)$ of the same loop, so two sweeps that end on different partners look like two different orbits as soon as the orbit is not a circle.
On the section at $T_a/2$ both partners give the same set $\{a(0), a(\pi)\}$: a period-1 orbit appears as **two** points, a period-2 orbit as four.
Between two recorded values, $g_2$ is ramped slowly (sub-steps of 20 kHz, $50\,T_a$ each): a direct jump of 0.2 MHz throws the trajectory out of the narrow basin of the satellite at $\kappa_b^{(0)}$ near $g_2^{\max}$, which a slow increase of $g_2$ does not do.
Sweeps, for each regime:
- **up from the cat**: $a(0) = i|\alpha|$, $b(0) = 0$ at 20 MHz, then up to 180 MHz;
- **down from the satellite**: the HB period-1 orbit at 137.4 MHz (just below $g_2^{\max}$), then down to 20 MHz;
- regime A, **up from the satellite**: the HB satellite at 45 MHz (above its fold at 40.4 MHz), then up to 180 MHz. This is the route of `torus_breakdown.ipynb`;
- regime B, **up from the satellite**: the HB satellite at 103 MHz (above its fold at 102.4 MHz), then up to 180 MHz.

The same script computes the largest Lyapunov exponent (section 3b) on three sweeps only, and the maps of section 4 (36 values of $\kappa_b$, steps of 0.5 MHz, $1000\,T_a$ per point).
The seeds (strobe states at $\tau = 0$ of the HB orbits) are written first. The Julia run (8 threads) is skipped if its output exists and `RUN_JULIA = False`.

In [ ]:
K_MAP = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.25, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.25, 9.5,
         9.75, 10.0, 10.5, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0]
G_DOWN = 137.4                                       # start of the down sweeps (g2_max = 137.44 MHz)
sat_gmax, z_k, k_prev = {}, z_hi.copy(), 1.0
for k in sorted(K_MAP, key=lambda k: (k < 1, abs(k - 1))):   # continue in kappa_b outward from kappa_b0
    z_k = orbit_at_kappa(k, z_hi if (k < 1) != (k_prev < 1) else z_k, 1.0 if (k < 1) != (k_prev < 1) else k_prev)
    k_prev = k
    hbk, z = HB(N_H, kfac=k, r=r0), z_k.copy()
    for s in np.linspace(s_hi, G_DOWN/g2_max, 40)[1:]:
        hbk.set_s(s); z = hbk.solve_robust(z)
    sat_gmax[k] = (strobe(z, N_H), np.abs(monodromy(hbk, z)).max())
print("period-1 orbit at 137.4 MHz, max|mu|: " + ", ".join(f"{k:g}: {sat_gmax[k][1]:.3f}" for k in K_MAP))

In [ ]:
z45 = z_sat60.copy()
for s in np.linspace(s60, 45/g2_max, 10)[1:]:
    hb.set_s(s); z45 = hb.solve(z45)
z749 = z_sat60.copy()                                # regime A, start of the torus_breakdown window
for s in np.linspace(s60, 74.9/g2_max, 20)[1:]:
    hb.set_s(s); z749 = hb.solve(z749)
hb8, z103 = HB(N_H, kfac=8.0, r=r0), orbit_at_kappa(8.0)        # regime B: satellite at 103 MHz (above its fold at 102.4 MHz)
for s in np.linspace(s_hi, 103/g2_max, 40)[1:]:
    hb8.set_s(s); z103 = hb8.solve_robust(z103)
print(f"satellite seeds: 74.9 MHz (kappa_b0) max|mu| = {np.abs(monodromy(hb, z749)).max():.3f}; "
      f"103 MHz (8 kappa_b0) max|mu| = {np.abs(monodromy(hb8, z103)).max():.3f}, memory phase {memory_phase(z103):+.3f} pi")

In [ ]:
SEEDS = OUT_DIR + "torus_crisis/beyond_cusp_seeds_22_10_2026.json"
json.dump({"description": "Strobe states (tau = 0, [x_a, x_b, v_a, v_b] in the rescaled flux form of the Hill notebooks) of the HB period-1 orbit, eps_d = 10.12 g2. "
                          "From beyond_cusp_regimes.ipynb, read by beyond_cusp_dynamics.jl",
           "g2_max_MHz": g2_max, "g_down_MHz": G_DOWN, "kappa_map": K_MAP, "kappa_diagrams": list(K_REG.values()),
           "satellite_at_g2max": {str(k): sat_gmax[k][0].tolist() for k in K_MAP},
           "satellite_45MHz_kappa1": strobe(z45, N_H).tolist(), "satellite_74.9MHz_kappa1": strobe(z749, N_H).tolist(),
           "satellite_103MHz_kappa8": strobe(z103, N_H).tolist()}, open(SEEDS, "w"))

In [ ]:
RUN_JULIA = False
DYN = OUT_DIR + "torus_crisis/beyond_cusp_dynamics_22_10_2026.json"
if RUN_JULIA or not os.path.exists(DYN):
    t0 = time.time()
    run = subprocess.run(["julia", "-t", "8", "--project=/Users/tmalas/Desktop/Dev/Simulations_local/Code_Julia", HERE + "beyond_cusp_dynamics.jl"],
                         capture_output=True, text=True)
    print(run.stdout[-3000:], run.stderr[-3000:])
    print(f"Julia run: {time.time() - t0:.0f} s")
D = json.load(open(DYN))
diag = {(d["kappa_over_kappa0"], d["sweep"]): d["data"] for d in D["diagrams"]}
maps = {(d["kappa_over_kappa0"], d["sweep"]): d["data"] for d in D["maps"]}
window = D["window"][0]["data"]
d0 = diag[(1.0, "up from the cat")]
print(f"check of the cat start at 20 MHz: |a| = {d0['abs_a_mean'][0]:.3f} (cat: |alpha| = {np.sqrt(r0/2):.3f}), period {d0['period_Ta'][0]} T_a")

In [ ]:
SWEEP_STYLE = {"up from the cat": C_CAT, "down from the satellite": C_SAT, "up from the satellite": "k"}

def hill_lines(ax, name):
    k = K_REG[name]
    ax.axvline(g2_max, color="gray", lw=0.6)
    ax.axvspan(g2_max, 185, color="0.6", alpha=0.12, lw=0)
    i = np.argmin(np.abs(k_fold - k))
    if abs(k_fold[i] - k) < 1e-9 and np.isfinite(g_sat_fold[i]):
        for g, c in ((g_sat_fold[i], C_SAT), (g_cat_fold[i], C_CAT)):
            ax.axvline(g, ls=":", color=c, lw=0.8)
    b = bubble[list(k_bub).index(k)] if k in k_bub else []
    for g in b:
        ax.axvline(g, ls="--", color=C_PD, lw=0.7)
    if name == "A":
        ax.axvline(g_ns_A, ls="-", color=C_NS, lw=0.7)
    j = [i for i, kk in enumerate(k_ns1) if abs(kk - k) < 1e-9]
    if j:
        ax.axvline(g_ns1[j[0]], ls="--", color=C_NS, lw=0.7)

fig, axes = plt.subplots(1, 3, figsize=(3.3*figsize_car[0], 1.1*figsize_car[1]))
for col, (name, k) in enumerate(K_REG.items()):
    ax = axes[col]
    for sw, c in SWEEP_STYLE.items():
        if (k, sw) not in diag:
            continue
        d = diag[(k, sw)]
        g = np.repeat(d["g2_MHz"], [len(x) for x in d["re_a"]])
        ax.scatter(g, np.abs(np.concatenate(d["re_a"]) + 1j*np.concatenate(d["im_a"])), s=0.15, color=c, lw=0, rasterized=True)
        ax.plot([], [], "s", ms=3, color=c, label=sw)
    hill_lines(ax, name)
    ax.set_title(rf"{name}: $\kappa_b = {k:g}\,\kappa_b^{{(0)}}$, {REG_NAME[name]}", fontsize=6)
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_xlim(20, 180)
axes[0].set_ylabel(r"$|a(nT_a/2)|$")
axes[0].legend(fontsize=4.5, loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR + "beyond_cusp_regimes_bifurcation_22_10_2026.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def classify(d):
    per = np.array(d["period_Ta"])
    return np.where(per == 1, "P1", np.where(per == 2, "P2", np.where(per > 2, "Pn", "aperiodic")))
for (k, sw), d in sorted(diag.items()):
    c, g = classify(d), np.array(d["g2_MHz"])
    order = np.argsort(g); c, g = c[order], g[order]
    segs, start = [], 0
    for i in range(1, len(c) + 1):
        if i == len(c) or c[i] != c[start]:
            if i - start >= 2 or c[start] == "aperiodic":
                segs.append(f"{c[start]} {g[start]:.1f}-{g[i - 1]:.1f}")
            start = i
    print(f"kappa_b x {k:g}, {sw}: " + ", ".join(segs))

Ordinate: $|a|$ at every drive period, the same set of points for the two partners $\pm a$: a period-1 orbit gives two points ($|a(0)|$ and $|a(\pi)|$, equal when the orbit is nearly a circle in the rotating frame, as the cat at low $g_2$).
Vertical lines (harmonic balance): dotted, satellite fold (olive) and cat fold (blue); dashed light blue, period-doubling bubble of the satellite; solid red, Neimark–Sacker of the period-2 orbit; dashed red, Neimark–Sacker of the period-1 orbit (formal); grey band, $g_2 > g_2^{\max}$.
The printed lists give the classes along each sweep: P1, P2, Pn = period 1, 2, $n > 2$ (up to 64) on the section at $T_a$; aperiodic = no period found (a torus, chaos, or an orbit still converging slowly).
Isolated points are omitted from the lists, except aperiodic ones.

## 3b. Bifurcation diagram and largest Lyapunov exponent, one sweep per regime
As in `torus_breakdown.ipynb` (diagram on top, $\lambda_1$ below). $\lambda_1$ from ChaosTools over $2000\,T_a$, in units of the code $\kappa_b$ of the run ($\kappa_b^{(0)}$, $8\,\kappa_b^{(0)}$ or $11\,\kappa_b^{(0)}$); in the article's convention $\lambda_1/\kappa_b$ is halved.
On a torus the finite-time $\lambda_1$ is slightly positive ($\propto 1/T$), so a point is called chaotic when $\lambda_1/\kappa_b > 0.1$, as in `torus_breakdown`.
- **A** ($\kappa_b^{(0)}$): the first figure of `torus_breakdown.ipynb`, redone with this model: satellite followed up from 74.9 to 79 MHz in steps of 20 kHz, $3000\,T_a$ per step, $\mathrm{Re}\,a(nT_a)$ over the last $600\,T_a$.
- **B** ($8\,\kappa_b^{(0)}$): satellite followed up from 103 MHz (the cat fold is not of interest here; the satellite carries the Neimark–Sacker point).
- **C** ($11\,\kappa_b^{(0)}$): the single branch, followed up from the cat.

In [ ]:
LAM_CHAOS = 0.1
# period doubling of the satellite at kappa_b0, resolved to 10 kHz (the bubble of section 1 is on a 0.55 MHz grid)
hbA, zA, prev = HB(N_H, r=r0), z749.copy(), None
for g in np.arange(74.9, 76.0, 0.01):
    hbA.set_s(g/g2_max); zA = hbA.solve(zA)
    mu = monodromy(hbA, zA); re = mu[np.abs(mu.imag) < 1e-9].real
    m = re.min() if len(re) else 0.0
    if prev is not None and prev > -1 >= m:
        g_pd_A = g - 0.01 + 0.01*(-1 - prev)/(m - prev)
        break
    prev = m

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(3.3*figsize_car[0], 1.7*figsize_car[1]), sharex="col", gridspec_kw={"height_ratios": [0.62, 0.38]})
# A: torus_breakdown window, Re a(nT_a)
gA = np.array(window["g2_MHz"])
reA = [np.array(x)[(len(x) - 1) % 2::2] for x in window["re_a"]]          # samples at multiples of T_a
imA = [np.array(x)[(len(x) - 1) % 2::2] for x in window["im_a"]]          # samples at multiples of T_a
axes[0, 0].scatter(np.repeat(gA, [len(x) for x in reA]), np.abs(np.concatenate(reA) + 1j*np.concatenate(imA)), s=0.15, color="k", lw=0, rasterized=True)
axes[1, 0].plot(gA, window["lambda1_over_kappa"], color="k", lw=1)
for ax in axes[:, 0]:
    ax.axvline(g_pd_A, ls="--", color=C_PD, lw=0.8, label="Hill: period doubling")
    ax.axvline(g_ns_A, ls="-", color=C_NS, lw=0.8, label="Hill: Neimark–Sacker (period 2)")
axes[0, 0].set_ylabel(r"$|a(nT_a)|$"); axes[0, 0].legend(fontsize=4.5, loc="lower left")
axes[1, 0].set_xlim(gA[0], gA[-1])
# B and C: |a| at every drive period
for col, (name, sw) in ((1, ("B", "up from the satellite")), (2, ("C", "up from the cat"))):
    d = diag[(K_REG[name], sw)]
    reT = [np.array(x)[(len(x) - 1) % 2::2] for x in d["re_a"]]    # samples at multiples of T_a
    imT = [np.array(x)[(len(x) - 1) % 2::2] for x in d["im_a"]]
    g = np.repeat(d["g2_MHz"], [len(x) for x in reT])
    axes[0, col].scatter(g, np.abs(np.concatenate(reT) + 1j*np.concatenate(imT)), s=0.15, color=SWEEP_STYLE[sw], lw=0, rasterized=True)
    axes[1, col].plot(d["g2_MHz"], d["lambda1_over_kappa"], color=SWEEP_STYLE[sw], lw=1)
    for ax in axes[:, col]:
        hill_lines(ax, name)
    axes[0, col].set_ylabel(r"$|a(nT_a)|$")
    axes[1, col].set_xlim(d["g2_MHz"][0] - 2, 180)
titles = {0: r"A: $\kappa_b^{(0)}$, satellite up (window of torus_breakdown)", 1: r"B: $8\,\kappa_b^{(0)}$, satellite up", 2: r"C: $11\,\kappa_b^{(0)}$, cat up"}
for col in range(3):
    axes[0, col].set_title(titles[col], fontsize=6)
    axes[1, col].axhline(0, color="gray", lw=0.5, ls=":")
    axes[1, col].axhline(LAM_CHAOS, color="gray", lw=0.5, ls="--")
    axes[1, col].set_xlabel(r"$g_2/2\pi$ (MHz)")
axes[1, 0].set_ylabel(r"$\lambda_1/\kappa_b$")
plt.tight_layout()
plt.savefig(FIG_DIR + "beyond_cusp_regimes_bifurcation_lyapunov_22_10_2026.png", dpi=300, bbox_inches="tight")
plt.savefig(FIG_DIR + "beyond_cusp_regimes_bifurcation_lyapunov_22_10_2026.pdf", bbox_inches="tight")
plt.show()

In [ ]:
perA, lamA = np.array(window["period_Ta"]), np.array(window["lambda1_over_kappa"])
i2, iap, ich = np.argmax(perA == 2), np.argmax((perA == 0) & (gA > gA[np.argmax(perA == 2)])), np.argmax(lamA > LAM_CHAOS)
print(f"A: period 2 from {gA[i2]:.2f} MHz (Hill {g_pd_A:.2f}), aperiodic from {gA[iap]:.2f} MHz (Hill NS {g_ns_A:.2f}), "
      f"lambda1 > {LAM_CHAOS} kappa_b from {gA[ich]:.2f} MHz; max lambda1/kappa_b on the torus window {lamA[(gA >= gA[iap]) & (gA < gA[ich])].max(initial=-np.inf):.3f}")
for name, sw in (("B", "up from the satellite"), ("C", "up from the cat")):
    d = diag[(K_REG[name], sw)]; g, lam = np.array(d["g2_MHz"]), np.array(d["lambda1_over_kappa"])
    print(f"{name}: max lambda1/kappa_b = {lam.max():+.3f} at {g[np.argmax(lam)]:.1f} MHz; points with lambda1/kappa_b > {LAM_CHAOS}: {np.sum(lam > LAM_CHAOS)}; "
          f"max below g2_max: {lam[g <= g2_max].max():+.3f}")

# 4. Type of attractor in the $(\kappa_b, g_2)$ plane
Same protocol as the diagrams on a coarser grid (36 values of $\kappa_b$, 0.5 MHz, slow ramp between points, $1000\,T_a$ at each point, period over $200\,T_a$), for the two sweeps:
slow increase of $g_2$ from the cat, and slow decrease from the satellite. Colour: period on the section at $T_a$. Without Lyapunov exponents, "aperiodic" does not separate a torus from chaos;
the bifurcation diagrams of section 3 show which it is for the three examples. Lines: those of section 1 (no fills).

In [ ]:
from matplotlib.colors import ListedColormap
CLASS_NAMES = ["period 1", "period 2", r"period $n > 2$", "aperiodic (torus or chaos)"]
CLASS_COLORS = ["0.88", C_PD, "#7b4fa0", C_NS]

def class_array(sweep, grid):
    Cl = np.full((len(K_MAP), len(grid)), np.nan)
    for i, k in enumerate(K_MAP):
        d = maps[(k, sweep)]
        g, per = np.array(d["g2_MHz"]), np.array(d["period_Ta"])
        c = np.where(per == 1, 0, np.where(per == 2, 1, np.where(per > 2, 2, 3)))
        o = np.argsort(g)
        j = np.clip(np.searchsorted(g[o], grid), 0, len(g) - 1)
        Cl[i] = np.where((grid >= g.min() - 1e-9) & (grid <= g.max() + 1e-9), c[o][j], np.nan)
    return Cl

g_grid = np.arange(20, 180.01, 0.5)
fig, axes = plt.subplots(1, 2, figsize=(2.6*figsize[0], 1.1*figsize[1]), sharey=True)
for ax, sw in zip(axes, ("up from the cat", "down from the satellite")):
    Cl = class_array(sw, g_grid)
    ax.pcolormesh(K_MAP, g_grid, Cl.T, cmap=ListedColormap(CLASS_COLORS), vmin=-0.5, vmax=3.5, shading="nearest", rasterized=True)
    regime_axes(ax, legend=False, lw=0.8, fills=False)
    for name, k in K_REG.items():
        ax.axvline(k, ls=":", color="k", lw=0.5); ax.text(k + 0.15, 22, name, fontsize=6)
    ax.set_xlim(0.4, 20.2); ax.set_ylim(20, 180); ax.set_title(sw, fontsize=7)
axes[1].set_ylabel("")
handles, labels = axes[0].get_legend_handles_labels()
handles += [mpl.patches.Patch(color=c) for c in CLASS_COLORS]; labels += CLASS_NAMES
fig.legend(handles, labels, fontsize=5, loc="upper center", bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False)
plt.tight_layout()
plt.savefig(FIG_DIR + "beyond_cusp_regimes_attractor_map_22_10_2026.pdf", bbox_inches="tight")
plt.show()

In [ ]:
print("aperiodic points (torus or chaos) per kappa_b:")
for k in K_MAP:
    parts = []
    for sw in ("up from the cat", "down from the satellite"):
        d = maps[(k, sw)]; g, per = np.array(d["g2_MHz"]), np.array(d["period_Ta"])
        ap = g[per == 0]
        parts.append(f"{sw}: {len(ap)} pts" + (f" ({ap.min():.1f}-{ap.max():.1f} MHz, {np.sum(ap <= g2_max)} below g2_max)" if len(ap) else ""))
    print(f"  kappa_b x {k:5.2f}: " + "; ".join(parts))

## Conclusions
All along $\varepsilon_d = 10.12\,g_2$; "formal" means $g_2 > g_2^{\max} = 137.4$ MHz.

**1. Regime diagram.** The Neimark–Sacker line of the period-2 orbit runs inside the period-doubling bubble of the satellite, from 76.3 MHz at $0.5\,\kappa_b^{(0)}$ to 96.4 MHz at $5.25\,\kappa_b^{(0)}$ (76.71 MHz at $\kappa_b^{(0)}$, as in `torus_breakdown` and `floquet_rescaling_kappa_b`), and ends with the bubble ($\kappa_b \simeq 5.36\,\kappa_b^{(0)}$, $g_2 \simeq 99$ MHz).

**2. Limit cycles (harmonic balance).**
- A ($\kappa_b^{(0)}$): cat stable up to its fold (97.2 MHz); satellite born at its fold (40.4 MHz), period-doubled between 75.4 and 104.0 MHz, stable again up to $g_2^{\max}$, Neimark–Sacker at 143 MHz (formal).
  The **period-2 orbit** is born at the period doubling, is stable only up to its Neimark–Sacker point at 76.71 MHz ($\arg\mu/2\pi = 0.103$ per $2T_a$), and stays unstable (complex pair, $\max|\mu|$ up to 5.5) up to 180 MHz.
- B ($8\,\kappa_b^{(0)}$): S between 102.4 and 105.8 MHz, no period doubling, stable up to $g_2^{\max}$, Neimark–Sacker at 158 MHz (formal).
- C ($11\,\kappa_b^{(0)}$): a single branch, stable up to $g_2^{\max}$, Neimark–Sacker at 169 MHz (formal).

**3. Bifurcation diagrams (section at every drive period, slow ramp).** With the section at $T_a/2$ the two partners give the same points: in C, the sweep up from the cat and the sweep down from the satellite fall on the same two curves ($|a(0)|$ and $|a(\pi)|$ of one orbit), which separate above 100 MHz where the orbit develops loops.
- A: from the satellite, period 1 → period 2 at 75.3 MHz → aperiodic from 76.7 MHz up to about 101.6 MHz, then a period-2 window (101.8–106.6 MHz) and alternating periodic windows and aperiodic bands (111–114, 116–120, 128.6–134 MHz), long periods up to 143 MHz, and a large period-1 orbit above (formal).
  Started on the cat, the trajectory stays on it up to the cat fold (97.2 MHz) and then falls into the same aperiodic set. Down from the satellite: period 1 down to 103.8 MHz, period 2, aperiodic from 101.0 down to 76.8 MHz, period 2, then the satellite again below 75.2 MHz.
- B: no aperiodic attractor in the physical range; hysteresis between the cat (jump at its fold, 105.8 MHz) and the satellite (jump at its fold, 102.4 MHz). Formally, a torus over 0.6 MHz above the Neimark–Sacker point (157 MHz), then a large period-1 orbit.
- C: a single period-1 attractor up to $g_2^{\max}$. Formally, a torus over 0.6 MHz above 168.4 MHz, then a large period-1 orbit.

**3b. Largest Lyapunov exponent.**
- A reproduces `torus_breakdown` with this model: period 2 from 75.26 MHz, torus from 76.70 MHz ($\lambda_1 \approx 0$, at most $0.024\,\kappa_b$), chaos from 77.24 MHz ($\lambda_1/\kappa_b \approx 0.6$–$1$).
- B (satellite up) and C (cat up): $\lambda_1 < 0$ everywhere in the physical range (at most $-0.19\,\kappa_b$ in B and $-0.03\,\kappa_b$ in C, the latter near 115 MHz where the orbit reshapes). $\lambda_1$ reaches 0 only at the formal Neimark–Sacker point, stays $\approx 0$ on the narrow torus, and becomes negative again on the large period-1 orbit. **No chaos.**

**4. Map.** Aperiodic attractors appear in the physical range only for $\kappa_b \le 5.25\,\kappa_b^{(0)}$, i.e. only where the satellite period-doubles (regime A). For $\kappa_b \ge 5.5\,\kappa_b^{(0)}$ both sweeps stay periodic up to $g_2^{\max}$; the only aperiodic points are 1–2 grid points just above the formal Neimark–Sacker line (the small torus).
So, along these sweeps, **chaos requires the period doubling of the satellite**: without it the period-1 attractor loses stability only formally, through a supercritical Neimark–Sacker bifurcation, and the system goes to a large periodic orbit, not to chaos.

**Caveats.** On the map, "aperiodic" does not separate a torus from chaos ($\lambda_1$ is computed on three sweeps only). Only the attractors reached by these sweeps are sampled; other attractors (e.g. the large orbits found in `Hill_method_satellite_fold` at $11\,\kappa_b^{(0)}$, 130 MHz) are not mapped.